# Recommendation 04 — Verifica della mappatura Qdrant `resource_id` ↔ MovieLens `movieId`

**Operazione in sola lettura.** Verifica la corrispondenza di **tutti** i punti della collezione GTE, con particolare attenzione al rischio che il notebook 02 abbia utilizzato l'**ID interno del punto Qdrant** anziché il `resource_id` presente nel payload.

Non esegue inferenza, non scarica i vettori, non modifica Qdrant, i notebook precedenti, i modelli MF o i dati di training. Legge i payload attraverso `scroll(..., with_payload=True, with_vectors=False)`.

**Interpretazione:** se per ogni punto l'ID scelto dalla *stessa logica del notebook 02* coincide con `resource_id`, e il catalogo e i titoli combaciano, la mappatura originale è compatibile con quella corretta. Se ci sono differenze **non utilizzare come definitive le metriche GTE precedenti**: bisogna rigenerare soltanto i ranking GTE leggendo i vettori esistenti mediante `resource_id` e ricalcolare le metriche, non gli embedding.

*Prerequisito*: `pip install qdrant-client pandas` se necessario.

## 0. Impostazioni

Copia in `notebooks/` del progetto, vicino al notebook `recommendation_02_baselines_mf_gte.ipynb`. Il controllo tenta di leggere il campo configurato in quel notebook (`QDRANT_MOVIE_ID_KEY`), per riprodurre la mappatura effettivamente impiegata. Verifica tale parametro manualmente, se nel progetto sono presenti più versioni.

In [1]:
from pathlib import Path
import ast, json, os, re, unicodedata
from collections import Counter
import pandas as pd
from IPython.display import display

PROJECT_ROOT_OVERRIDE = None  # es. Path('/Users/.../progetto')
PROTOCOL_ID = 'rs_coldstart_v1'
QDRANT_URL = 'http://localhost:6333'
QDRANT_COLLECTION = 'movies_d_full_enriched_v2_gte_multilingual_base_v1'
QDRANT_API_KEY = None  # oppure variabile d'ambiente QDRANT_API_KEY
RESOURCE_ID_FIELD = 'resource_id'

# Forza questa stringa solo se la copia del notebook 02 effettivamente eseguita
# aveva una diversa impostazione QDRANT_MOVIE_ID_KEY.
ORIGINAL_CONFIG_KEY_OVERRIDE = None

# Riporta gli errori su disco senza toccare gli artefatti del benchmark.
SAVE_REPORTS = True


def find_root():
    if PROJECT_ROOT_OVERRIDE is not None:
        return Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
    current=Path.cwd().resolve()
    for candidate in [current,*current.parents]:
        if (candidate/'datasets/processed/recommendation_protocol'/PROTOCOL_ID/'protocol_manifest.json').is_file():
            return candidate
    raise FileNotFoundError('Non trovo protocol_manifest.json. Imposta PROJECT_ROOT_OVERRIDE')

ROOT=find_root()
BASE=ROOT/'datasets/processed/recommendation_protocol'/PROTOCOL_ID
REPORT=BASE/'evaluations'/'qdrant_resource_id_check'
CATALOG_PATH=BASE/'inputs'/'catalog_movie_ids.csv'
MOVIES_PATH=ROOT/'datasets/raw/ml-latest-small/movies.csv'
MODEL_NB_PATHS=[ROOT/'notebooks/recommendation_02_baselines_mf_gte.ipynb',
                ROOT/'recommendation_02_baselines_mf_gte.ipynb',
                *sorted((ROOT/'notebooks').glob('recommendation_02_baselines_mf_gte*.ipynb'))]
MODEL_NOTEBOOK=next((p for p in MODEL_NB_PATHS if p.is_file()),None)


def find_original_key(path):
    if path is None:raise FileNotFoundError('Non trovo notebook 02: usa ORIGINAL_CONFIG_KEY_OVERRIDE')
    raw=json.loads(path.read_text(encoding='utf-8'))
    found=[]
    for c in raw.get('cells',[]):
        if c.get('cell_type')!='code':continue
        source=''.join(c.get('source',[]))
        for line in source.splitlines():
            m=re.match(r'^\s*QDRANT_MOVIE_ID_KEY\s*=\s*([\"\'])(.*?)\1',line)
            if m:found.append(m.group(2))
    if not found:raise ValueError(f'QDRANT_MOVIE_ID_KEY non trovato in {path}. Imposta override.')
    return found[-1]

ORIGINAL_KEY=ORIGINAL_CONFIG_KEY_OVERRIDE or find_original_key(MODEL_NOTEBOOK)
expected_df=pd.read_csv(CATALOG_PATH)
if 'movieId' not in expected_df.columns:raise ValueError('Colonna movieId assente dal catalogo')
expected=set(expected_df.movieId.astype(int))
movies=pd.read_csv(MOVIES_PATH)
if not {'movieId','title'}.issubset(movies.columns):raise ValueError('movies.csv senza movieId/title')
movies_by_id=dict(zip(movies.movieId.astype(int),movies.title.astype(str)))
print('Collection:',QDRANT_COLLECTION)
print('Notebook 02 individuato:',MODEL_NOTEBOOK)
print('Chiave originale simulata:',ORIGINAL_KEY)
print('Chiave canonica:',RESOURCE_ID_FIELD)
print('Film attesi:',len(expected))
print('Nota: se la copia del notebook 02 effettivamente eseguita aveva una chiave diversa, usare ORIGINAL_CONFIG_KEY_OVERRIDE.')

Collection: movies_d_full_enriched_v2_gte_multilingual_base_v1
Notebook 02 individuato: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/notebooks/recommendation_02_baselines_mf_gte.ipynb
Chiave originale simulata: movieId
Chiave canonica: resource_id
Film attesi: 1996
Nota: se la copia del notebook 02 effettivamente eseguita aveva una chiave diversa, usare ORIGINAL_CONFIG_KEY_OVERRIDE.


## 1. Funzioni di controllo e mappatura originaria

La funzione `old_mapping` riproduce **l'ordine di precedenza del caricatore originale**: campo configurato `QDRANT_MOVIE_ID_KEY` → `movie_id` / `movieId_ml` → ID interno del punto Qdrant quando numerico e incluso nel catalogo. Non assume che `id` e `resource_id` siano equivalenti.

In [2]:
def strict_int(x, what):
    if isinstance(x,bool) or x is None:raise ValueError(f'{what} non è un intero: {x!r}')
    if isinstance(x,float) and not x.is_integer():raise ValueError(f'{what} contiene decimali: {x}')
    if isinstance(x,str) and not re.fullmatch(r'[+-]?\d+',x.strip()):
        raise ValueError(f'{what} non è un intero: {x!r}')
    return int(x)


def old_mapping(point_id,payload,configured_key,allowed_ids):
    value=payload.get(configured_key)
    method=f'payload.{configured_key}' if value is not None else None
    if value is None:
        if payload.get('movie_id') is not None:
            value=payload['movie_id']; method='payload.movie_id'
        elif payload.get('movieId_ml') is not None:
            value=payload['movieId_ml']; method='payload.movieId_ml'
    if value is None:
        if isinstance(point_id,int) and not isinstance(point_id,bool) and point_id in allowed_ids:
            value=point_id;method='point.id'
        else:
            return None,'FAILED_NO_MATCH'
    try:return strict_int(value,'old id'),method
    except ValueError:return None,'FAILED_INVALID_ID'


def normalize_title(title):
    text=unicodedata.normalize('NFKD',str(title))
    text=''.join(c for c in text if not unicodedata.combining(c))
    text=re.sub(r'\s*\(\d{4}\)\s*$','',text).casefold().strip()
    return re.sub(r'[^\w]+',' ',text).strip()


def classify_point(p):
    payload=p.payload or {}
    actual=str(payload.get(RESOURCE_ID_FIELD,''))
    try:rid=strict_int(payload.get(RESOURCE_ID_FIELD),RESOURCE_ID_FIELD);rerr=None
    except Exception as e:rid=None;rerr=str(e)
    oid,method=old_mapping(p.id,payload,ORIGINAL_KEY,expected)
    expected_title=movies_by_id.get(rid)
    payload_title=payload.get('title')
    title_match=(normalize_title(expected_title)==normalize_title(payload_title)) if expected_title and payload_title else None
    return {
        'point_id':str(p.id),'resource_id':rid,'original_mapped_movieId':oid,
        'original_resolution_method':method,'id_match':rid is not None and oid==rid,
        'in_catalog':rid in expected if rid is not None else False,
        'original_in_catalog':oid in expected if oid is not None else False,
        'payload_title':payload_title,'movielens_title':expected_title,'title_normalized_equal':title_match,
        'resource_id_error':rerr,'embedding_model':payload.get('embedding_model'),
        'schema_version':payload.get('schema_version')
    }

# Test isolati senza Qdrant
class Stub:
    def __init__(self,id,payload):self.id=id;self.payload=payload
assert classify_point(Stub(2,{'resource_id':2,'title':'Jumanji'}))['id_match']  # film 2
assert not classify_point(Stub(99,{'resource_id':2,'title':'Jumanji'}))['id_match'] if ORIGINAL_KEY not in ('resource_id',) else True
print('Verificata la funzione di confronto su esempi controllati.')

Verificata la funzione di confronto su esempi controllati.


## 2. Scansione Qdrant in sola lettura

Questo è l'unico passaggio che effettua chiamate di rete. Si leggono **solo payload e identificativi dei punti**, non i vettori. Non sono necessari calcoli GTE.

In [3]:
from qdrant_client import QdrantClient
settings={'url':QDRANT_URL,'timeout':60}
key=QDRANT_API_KEY or os.environ.get('QDRANT_API_KEY')
if key:settings['api_key']=key
client=QdrantClient(**settings)
info=client.get_collection(QDRANT_COLLECTION)
rows=[]
offset=None
iterations=0
while True:
    points,offset=client.scroll(collection_name=QDRANT_COLLECTION,offset=offset,limit=256,
                                with_payload=True,with_vectors=False)
    rows.extend([classify_point(p) for p in points])
    iterations+=1
    if offset is None:break
    if iterations>10000:raise RuntimeError('Troppi batch Qdrant. Interrotta scansione.')

df=pd.DataFrame(rows)
if df.empty:raise ValueError('La collezione non contiene punti')
print('Qdrant points_count:',info.points_count,'| punti letti:',len(df))
print('Metodi mappatura notebook 02:')
display(df.original_resolution_method.value_counts().rename_axis('metodo').reset_index(name='numero_punti'))

invalid=df[df.resource_id.isna()]
missing=sorted(expected-set(df.resource_id.dropna().astype(int)))
unexpected=sorted(set(df.resource_id.dropna().astype(int))-expected)
duplicates=df[df.resource_id.duplicated(keep=False) & df.resource_id.notna()]
original_duplicate=df[df.original_mapped_movieId.duplicated(keep=False) & df.original_mapped_movieId.notna()]
wrong=df[df.id_match!=True]
unmatched_titles=df[df.title_normalized_equal==False]

summary={
 'collection':QDRANT_COLLECTION,'points_read':int(len(df)),
 'catalog_size':len(expected),'id_matches':int(df.id_match.sum()),
 'id_mismatches_or_missing':int(len(wrong)),
 'missing_resource_ids':missing,'unexpected_resource_ids':unexpected,
 'duplicated_resource_ids':int(len(duplicates)),
 'original_duplicate_movieIds':int(len(original_duplicate)),
 'title_mismatches_after_normalization':int(len(unmatched_titles)),
 'original_config_key':ORIGINAL_KEY,
 'method_counts':{str(k):int(v) for k,v in df.original_resolution_method.value_counts().items()},
 'metadata_embedding_models':{str(k):int(v) for k,v in df.embedding_model.fillna('NULL').value_counts().items()},
 'metadata_schema_versions':{str(k):int(v) for k,v in df.schema_version.fillna('NULL').value_counts().items()}
}
core_ok=(len(df)==len(expected) and not missing and not unexpected and duplicates.empty and
         original_duplicate.empty and wrong.empty)
summary['core_mapping_ok']=bool(core_ok)
print('RISULTATO CORE:', 'PASS — mapping originale coincide con resource_id per tutti i film' if core_ok else
      'FAIL — MAPPATURA NON CERTIFICATA: controllare differenze prima di usare le metriche GTE')
print('Differenze di identificativo:',len(wrong),'| Mancanti:',len(missing),
      '| Duplicati:',len(duplicates),'| Differenze titoli:',len(unmatched_titles))
if len(wrong):display(wrong.head(25))
if len(unmatched_titles):
    print('Attenzione: la normalizzazione dei titoli è solo un indizio, non prova automatica di errore.')
    display(unmatched_titles[['point_id','resource_id','payload_title','movielens_title']].head(25))

if SAVE_REPORTS:
    REPORT.mkdir(parents=True,exist_ok=True)
    df.to_csv(REPORT/'qdrant_id_full_audit.csv',index=False)
    wrong.to_csv(REPORT/'qdrant_id_mismatches.csv',index=False)
    unmatched_titles.to_csv(REPORT/'qdrant_title_mismatches.csv',index=False)
    (REPORT/'qdrant_id_audit_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
    print('Report:',REPORT)

Qdrant points_count: 1996 | punti letti: 1996
Metodi mappatura notebook 02:


,metodo,numero_punti
0,point.id,1996


RISULTATO CORE: PASS — mapping originale coincide con resource_id per tutti i film
Differenze di identificativo: 0 | Mancanti: 0 | Duplicati: 0 | Differenze titoli: 540
Attenzione: la normalizzazione dei titoli è solo un indizio, non prova automatica di errore.


,point_id,resource_id,payload_title,movielens_title
8,11,11,The American President,"American President, The (1995)"
23,29,29,The City of Lost Children,"City of Lost Children, The (Cité des enfants p..."
25,32,32,Twelve Monkeys,Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
33,47,47,Se7en,Seven (a.k.a. Se7en) (1995)
35,50,50,The Usual Suspects,"Usual Suspects, The (1995)"
37,58,58,The Postman,"Postman, The (Postino, Il) (1994)"
38,60,60,The Indian in the Cupboard,"Indian in the Cupboard, The (1995)"
43,73,73,Les Miserables,"Misérables, Les (1995)"
45,79,79,The Juror,"Juror, The (1996)"
53,105,105,The Bridges of Madison County,"Bridges of Madison County, The (1995)"


Report: /Users/danieleraveglia/Documents/uniMi/TESI/progetto/datasets/processed/recommendation_protocol/rs_coldstart_v1/evaluations/qdrant_resource_id_check


## 3. Come interpretare l'esito

- **PASS**: tutti i 1.996 film attesi sono presenti, `resource_id` è unico, e il `movieId` ricostruito **con la logica originaria del notebook 02** è identico a `resource_id` per ogni punto. La mappatura, per questa collezione, non è la spiegazione del vantaggio di Popularity.
- **FAIL**: il notebook 02 potrebbe avere associato vettori GTE a film differenti; occorre correggere `QDRANT_MOVIE_ID_KEY='resource_id'`, rigenerare **solo le predizioni GTE** e ricalcolare le metriche (usando gli stessi split e senza ricalcolare gli embedding). Non modificare MF/Popularity.
- **Titoli discordanti ma core PASS**: verificare i titoli prima di trarre conclusioni; versioni, titoli internazionali e differenze di naming possono essere innocue. Se ci sono molti titoli discordanti, ricontrollare anche l'ingestione.
- La scansione è **soltanto sui payload**: non dimostra che il vettore incorpori il testo di *quel* film, né risolve il potenziale leakage dei tag MovieLens nelle descrizioni.

**Invia `qdrant_id_audit_summary.json`**; se `core_mapping_ok=false`, anche `qdrant_id_mismatches.csv`.